# Match Results Predictor

> In this Jupiter Notebook, I will used Advanced AI models in order to predict football match results.


### 1 Data Loading and Preperation

In [ ]:
import pandas as pd
import glob
import os

# Columns to keep

KEEP_COLS = ["Div", "Date", "HomeTeam", "AwayTeam", "FTHG", "FTAG", "FTR", "HTHG", "HTAG", "HTR", "Attendance", "Referee", "HS", "AS", "HST", "AST", "HHW", "AHW", "HC", "AC", "HF", "AF", "HO", "AO", "HY", "AY", "HR", "AR", "HBP", "ABP"]

# Load and combine all CSVs 

CSV_DIR = "../Data/Premier League"

all_files = sorted(glob.glob(os.path.join(CSV_DIR, "*.csv")))
print(f"Found {len(all_files)} CSV files")

dfs = []
for path in all_files:
    season = os.path.basename(path).replace(".csv", "")
    for encoding in ["utf-8-sig", "latin-1", "windows-1252"]:
        try:
            df = pd.read_csv(
                path,
                encoding=encoding,
                on_bad_lines="skip",
                engine="python",
            )
            break
        except UnicodeDecodeError:
            continue
    df["Season"] = season
    dfs.append(df)

print(f"\nLoaded {len(dfs)} files successfully")

raw = pd.concat(dfs, ignore_index=True)
print(f"Total rows: {len(raw)}")

# Keep only relevant columns, filling missing ones with NaN

data = raw.reindex(columns=KEEP_COLS + ["Season"])

# Basic cleanup

def parse_date(val):
    for fmt in ["%d/%m/%Y", "%d/%m/%y"]:
        try:
            return pd.to_datetime(val, format=fmt)
        except:
            pass
    return pd.NaT

data["Date"] = data["Date"].apply(parse_date)

data = data.dropna(subset=["FTR"])

data = data.sort_values("Date").reset_index(drop=True)

# Summary

print(f"\nMissing values per column:")
missing = data.isnull().sum()
missing = missing[missing > 0]
if missing.empty:
    print("  None!")
else:
    for col, count in missing.items():
        pct = count / len(data) * 100
        print(f"  {col:<10} {count:>5} missing  ({pct:.1f}%)")

Found 26 CSV files

Loaded 26 files successfully
Total rows: 9720

Missing values per column:
  Attendance  8960 missing  (92.2%)
  HHW         8959 missing  (92.2%)
  AHW         8959 missing  (92.2%)
  HO          8959 missing  (92.2%)
  AO          8959 missing  (92.2%)
  HBP         8959 missing  (92.2%)
  ABP         8959 missing  (92.2%)


### 2 Adding Gx Columns

The meaning of xG in football is “expected goals.” It’s a statistic used to measure how likely a shot is to become a goal.

Each shot gets a value between 0 and 1 based on factors like:

distance from goal
angle
whether it was a header or a shot with the foot
whether it was a counterattack
how many defenders were nearby
the type of pass before the shot

In [ ]:
import asyncio
import aiohttp
import understat
import pandas as pd
from tqdm import tqdm

# Fetch xG data from Understat
# Understat covers Premier League from 2014/15 onwards
# Seasons before that will have NaN for xG columns

SEASONS = list(range(2014, 2026))

async def fetch_season(session, year):
    u = understat.Understat(session)
    matches = await u.get_league_results("EPL", year)
    rows = []
    for m in matches:
        rows.append({
            "Date"     : m["datetime"][:10],
            "HomeTeam" : m["h"]["title"],
            "AwayTeam" : m["a"]["title"],
            "xG_home"  : float(m["xG"]["h"]),
            "xG_away"  : float(m["xG"]["a"]),
        })
    return rows

async def fetch_all():
    all_rows = []
    async with aiohttp.ClientSession() as session:
        for year in tqdm(SEASONS, desc="Fetching xG"):
            try:
                rows = await fetch_season(session, year)
                all_rows.extend(rows)
                print(f"  {year}/{str(year+1)[-2:]}  →  {len(rows)} matches")
            except Exception as e:
                print(f"  {year}  →  FAILED: {e}")
    return all_rows

# Running the async fetcher (works in Jupyter with nest_asyncio)
import nest_asyncio
nest_asyncio.apply()

rows = asyncio.run(fetch_all())
xg_df = pd.DataFrame(rows)
xg_df["Date"] = pd.to_datetime(xg_df["Date"])

print(f"\nFetched {len(xg_df)} xG records")

# Normalise team names to match football-data.co.uk because understat uses different team names

NAME_MAP = {
    "Manchester United"  : "Man United",
    "Manchester City"    : "Man City",
    "Newcastle United"   : "Newcastle",
    "Tottenham"          : "Tottenham",
    "Wolverhampton"      : "Wolves",
    "Sheffield United"   : "Sheffield United",
    "West Bromwich"      : "West Brom",
    "Nottingham Forest"  : "Nott'm Forest",
    "Leeds"              : "Leeds",
    "Leicester"          : "Leicester",
}

xg_df["HomeTeam"] = xg_df["HomeTeam"].replace(NAME_MAP)
xg_df["AwayTeam"] = xg_df["AwayTeam"].replace(NAME_MAP)

# Merge into main dataset

data = pd.merge(
    data,
    xg_df,
    on=["Date", "HomeTeam", "AwayTeam"],
    how="left",   # keep all rows, NaN for seasons before 2014/15
)

# Summary

total     = len(data)
with_xg   = data["xG_home"].notna().sum()
without   = total - with_xg

print(f"\n{'─'*45}")
print(f"Total matches : {total:,}")
print(f"With xG       : {with_xg:,}  ({with_xg/total*100:.1f}%)")
print(f"Without xG    : {without:,}")

data[["Date", "HomeTeam", "AwayTeam", "FTR", "xG_home", "xG_away"]].tail(10)
data.to_csv("../Data/combined_data.csv", index=False)

Fetching xG:   8%|▊         | 1/12 [00:01<00:13,  1.19s/it]

  2014/15  →  380 matches


Fetching xG:  17%|█▋        | 2/12 [00:01<00:08,  1.18it/s]

  2015/16  →  380 matches


Fetching xG:  25%|██▌       | 3/12 [00:02<00:06,  1.37it/s]

...

  2023/24  →  380 matches


Fetching xG:  92%|█████████▏| 11/12 [00:05<00:00,  2.72it/s]

  2024/25  →  380 matches


Fetching xG: 100%|██████████| 12/12 [00:05<00:00,  2.23it/s]

  2025/26  →  349 matches

Fetched 4529 xG records

─────────────────────────────────────────────
Total matches : 9,719
With xG       : 3,945  (40.6%)
Without xG    : 5,774


### 3 Adding Sentiment Columns using an LLM

In [3]:
from transformers import pipeline

print("Loading sentiment model...")
sentiment_model = pipeline(
    "sentiment-analysis",
    model="cardiffnlp/twitter-roberta-base-sentiment",
    truncation=True,
    max_length=512
)
print("Model loaded!")

c:\Users\angel\Files\UCLL\Semester 4\Advanced AI\Match Results Project\VSC - Project\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading sentiment model...


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 30012.64it/s]


Model loaded!


In [ ]:
import requests
import pandas as pd
import time
from datetime import timedelta

GUARDIAN_API_KEY = "8289721f-2ef8-4cd3-b2a3-bd48ba877685"

data["Date"] = pd.to_datetime(data["Date"])
recent = data[data["Date"].dt.year >= 2020].copy().reset_index(drop=True)

# Load checkpoint if exists (so you can resume if interrupted)
import os
CHECKPOINT = "../Data/sentiment_checkpoint.csv"
if os.path.exists(CHECKPOINT):
    checkpoint_df = pd.read_csv(CHECKPOINT)
    checkpoint_df["Date"] = pd.to_datetime(checkpoint_df["Date"])
    done_indices = set(checkpoint_df.index)
    print(f"Resuming from checkpoint — {len(done_indices)} already done")
else:
    checkpoint_df = recent.copy()
    checkpoint_df["home_sentiment"] = None
    checkpoint_df["away_sentiment"] = None
    done_indices = set()
    print("Starting fresh")

# Fetch headlines from Guardian API

def fetch_guardian_headlines(team, match_date, days_before=5):
    date_from = (match_date - timedelta(days=days_before)).strftime("%Y-%m-%d")
    date_to   = match_date.strftime("%Y-%m-%d")
    try:
        r = requests.get(
            "https://content.guardianapis.com/search",
            params={
                "q"           : f"{team} football",
                "section"     : "football",
                "from-date"   : date_from,
                "to-date"     : date_to,
                "page-size"   : 5,
                "order-by"    : "relevance",
                "show-fields" : "headline,trailText",
                "api-key"     : GUARDIAN_API_KEY,
            },
            timeout=10
        )
        results = r.json().get("response", {}).get("results", [])
        texts = []
        for a in results:
            fields = a.get("fields", {})
            headline = fields.get("headline", a.get("webTitle", ""))
            trail    = fields.get("trailText", "")
            texts.append(f"{headline}. {trail}".strip())
        return texts if texts else None
    except Exception:
        return None

# Score sentiment with HuggingFace model

LABEL_SCORE = {"LABEL_0": -1.0, "LABEL_1": 0.0, "LABEL_2": 1.0}

def score_headlines(texts):
    if not texts:
        return None
    try:
        results = sentiment_model(texts)
        scores  = [LABEL_SCORE[r["label"]] * r["score"] for r in results]
        return round(sum(scores) / len(scores), 4)
    except Exception:
        return None

# Process all matches

for i, row in checkpoint_df.iterrows():
    if pd.notna(checkpoint_df.at[i, "home_sentiment"]):
        continue  # already processed, skip

    home_texts = fetch_guardian_headlines(row["HomeTeam"], row["Date"])
    away_texts = fetch_guardian_headlines(row["AwayTeam"], row["Date"])

    checkpoint_df.at[i, "home_sentiment"] = score_headlines(home_texts)
    checkpoint_df.at[i, "away_sentiment"] = score_headlines(away_texts)

    # Save checkpoint every 50 matches
    if i % 50 == 0:
        checkpoint_df.to_csv(CHECKPOINT, index=False)
        done = i + 1
        h = checkpoint_df.at[i, "home_sentiment"]
        print(f"  {done}/{len(recent)}  {row['HomeTeam']} vs {row['AwayTeam']}  → home: {h}")

    time.sleep(0.5)

# Final save & merge

checkpoint_df.to_csv(CHECKPOINT, index=False)

data = data.merge(
    checkpoint_df[["Date", "HomeTeam", "AwayTeam", "home_sentiment", "away_sentiment"]],
    on=["Date", "HomeTeam", "AwayTeam"],
    how="left"
)

# Summary

total     = len(data)
with_sent = data["home_sentiment"].notna().sum()
print(f"\n{'─'*45}")
print(f"Total matches     : {total:,}")
print(f"With sentiment    : {with_sent:,}  ({with_sent/total*100:.1f}%)")
print(f"\nSample scores:")
print(data[data["home_sentiment"].notna()][
    ["Date","HomeTeam","AwayTeam","home_sentiment","away_sentiment"]
].head(10).to_string())

data.to_csv("../Data/combined_data.csv", index=False)
if os.path.exists(CHECKPOINT):
    os.remove(CHECKPOINT)
print("Done!")

Resuming from checkpoint — 2390 already done
  751/2390  Chelsea vs Liverpool  → home: 0.3103
  801/2390  Liverpool vs Norwich  → home: 0.4236
  851/2390  Burnley vs Man City  → home: 0.5057
  901/2390  West Ham vs Arsenal  → home: -0.2394
  951/2390  Man United vs Brighton  → home: -0.1373
  1001/2390  Man United vs Arsenal  → home: -0.1454
  1051/2390  Chelsea vs Man United  → home: 0.3347
  1101/2390  Brighton vs Arsenal  → home: 0.3071
  1151/2390  Man United vs Leeds  → home: 0.1026
  1201/2390  West Ham vs Aston Villa  → home: -0.3806
  1251/2390  Crystal Palace vs Everton  → home: -0.1387
  1301/2390  Bournemouth vs Man United  → home: 0.0028
  1351/2390  Luton vs West Ham  → home: -0.1187
  1401/2390  West Ham vs Newcastle  → home: -0.1197
  1451/2390  Fulham vs Wolves  → home: 0.0669
  1501/2390  Bournemouth vs Fulham  → home: nan
  1551/2390  Wolves vs Brentford  → home: nan
  1601/2390  Burnley vs Brentford  → home: nan
  1651/2390  Fulham vs Liverpool  → home: nan


KeyboardInterrupt: 

In [6]:
import requests

r = requests.get(
    "https://content.guardianapis.com/search",
    params={
        "q"       : "Arsenal football",
        "section" : "football",
        "page-size": 1,
        "api-key" : GUARDIAN_API_KEY,
    }
)

response = r.json().get("response", {})
status   = response.get("status")
pages    = response.get("pages")

if status == "ok":
    print("API is working — quota refreshed, good to go!")
    print(f"   Test query returned {pages} pages of results")
else:
    print("API not working yet")
    print(f"   Response: {r.json()}")

API not working yet
   Response: {'message': 'API rate limit exceeded'}


### 4 Clean & Engineer Features

In [ ]:
import pandas as pd
import numpy as np

# Step 1: Filter to 2014 onwards (where xG exists)

data["Date"] = pd.to_datetime(data["Date"])
data = data[data["Date"].dt.year >= 2014].copy().reset_index(drop=True)
print(f"Matches from 2014 onwards: {len(data)}")

# Step 2: Drop useless columns

DROP_COLS = [
    "Div", "Season", "Referee", "Attendance",  # irrelevant
    "HHW", "AHW", "HO", "AO", "HBP", "ABP",    # 90%+ missing
    "HTHG", "HTAG", "HTR",                     # half time, not available pre match
    "FTHG", "FTAG",                            # these are the result so we don't want to leak them
    "HS", "AS", "HST", "AST",                  # post match stats, not available pre-match
    "HC", "AC", "HF", "AF",                    # post match stats
    "HY", "AY", "HR", "AR",                    # post match stats
]
data = data.drop(columns=[c for c in DROP_COLS if c in data.columns])
print(f"Columns remaining: {list(data.columns)}")

# Step 3: Encode target variable

data["target"] = data["FTR"].map({"H": 0, "D": 1, "A": 2})
data = data.drop(columns=["FTR"])

# Step 4: Feature Engineering
# Important: only use data available before the match (no leakage)
# We calculate rolling stats over the last 5 games for each team

data = data.sort_values("Date").reset_index(drop=True)

def rolling_team_stats(df, team_col, goals_for_col, goals_against_col, xg_col, result_col, prefix, n=5):
    """Calculate rolling averages for a team over last N matches."""
    results = []
    for idx, row in df.iterrows():
        team      = row[team_col]
        match_date = row["Date"]

        # Get all previous matches for this team (home or away)
        home_mask = (df["HomeTeam"] == team) & (df["Date"] < match_date)
        away_mask = (df["AwayTeam"] == team) & (df["Date"] < match_date)

        home_matches = df[home_mask].tail(n)
        away_matches = df[away_mask].tail(n)

        # Combine and take last N
        all_prev = pd.concat([home_matches, away_matches]).sort_values("Date").tail(n)

        if len(all_prev) == 0:
            results.append({
                f"{prefix}_form"    : np.nan,
                f"{prefix}_gf_avg"  : np.nan,
                f"{prefix}_ga_avg"  : np.nan,
                f"{prefix}_xg_avg"  : np.nan,
            })
            continue

        # Form: points per game (W=3, D=1, L=0)
        def get_points(r, team):
            if r["FTR"] == "H": return 3 if r["HomeTeam"] == team else 0
            if r["FTR"] == "A": return 0 if r["HomeTeam"] == team else 3
            return 1

        # Re-load FTR temporarily for form calculation
        prev_with_ftr = df.loc[all_prev.index].copy()
        points = prev_with_ftr.apply(lambda r: get_points(r, team), axis=1)
        form   = points.mean()

        # Goals for/against
        gf = pd.concat([
            df.loc[home_mask, "FTHG"].tail(n),
            df.loc[away_mask, "FTAG"].tail(n)
        ]).mean()

        ga = pd.concat([
            df.loc[home_mask, "FTAG"].tail(n),
            df.loc[away_mask, "FTHG"].tail(n)
        ]).mean()

        # xG average
        xg = pd.concat([
            df.loc[home_mask, "xG_home"].tail(n),
            df.loc[away_mask, "xG_away"].tail(n)
        ]).mean()

        results.append({
            f"{prefix}_form"   : round(form, 4),
            f"{prefix}_gf_avg" : round(gf,   4),
            f"{prefix}_ga_avg" : round(ga,   4),
            f"{prefix}_xg_avg" : round(xg,   4),
        })

    return pd.DataFrame(results, index=df.index)

# We need FTR temporarily for form calculation, add it back
data["FTR"] = data["target"].map({0: "H", 1: "D", 2: "A"})

print("Engineering home team features...")
home_features = rolling_team_stats(data, "HomeTeam", "FTHG", "FTAG", "xG_home", "FTR", "home")

print("Engineering away team features...")
away_features = rolling_team_stats(data, "AwayTeam", "FTAG", "FTHG", "xG_away", "FTR", "away")

data = pd.concat([data, home_features, away_features], axis=1)
data = data.drop(columns=["FTR", "FTHG", "FTAG"], errors="ignore")

# Step 5: Add head to head feature 

def h2h_form(df, idx, home_team, away_team, n=5):
    """Home team win rate in last N meetings vs this away team."""
    mask = (
        ((df["HomeTeam"] == home_team) & (df["AwayTeam"] == away_team)) |
        ((df["HomeTeam"] == away_team) & (df["AwayTeam"] == home_team))
    ) & (df["Date"] < df.loc[idx, "Date"])
    prev = df[mask].tail(n)
    if len(prev) == 0:
        return np.nan
    wins = sum(
        1 for _, r in prev.iterrows()
        if (r["HomeTeam"] == home_team and r["target"] == 0) or
           (r["AwayTeam"] == home_team and r["target"] == 2)
    )
    return round(wins / len(prev), 4)

data["FTR"] = data["target"].map({0: "H", 1: "D", 2: "A"})
print("Engineering head-to-head features...")
data["h2h_home_winrate"] = [
    h2h_form(data, i, row["HomeTeam"], row["AwayTeam"])
    for i, row in data.iterrows()
]
data = data.drop(columns=["FTR"], errors="ignore")

# Step 6: Final cleanup

# Fill sentiment NaN with 0 (neutral) for matches without news data
if "home_sentiment" in data.columns:
    data["home_sentiment"] = data["home_sentiment"].fillna(0)
    data["away_sentiment"] = data["away_sentiment"].fillna(0)

# Drop rows where we have no rolling stats (first few matches of each team)
feature_cols = [c for c in data.columns if c not in ["Date", "HomeTeam", "AwayTeam", "target"]]
data = data.dropna(subset=["home_form", "away_form"]).reset_index(drop=True)

# Fill remaining NaN with column median
for col in feature_cols:
    if data[col].isnull().any():
        data[col] = data[col].fillna(data[col].median())

print(f"\nFinal dataset: {len(data)} matches")
print(f"Features: {feature_cols}")
print(f"\nTarget distribution:")
print(data["target"].value_counts().sort_index().rename({0:"Home Win", 1:"Draw", 2:"Away Win"}))

data.to_csv("../Data/model_data.csv", index=False)
print("\nSaved to model_data.csv!")

### 5 Neural Network

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np
import pandas as pd

# Prepare features & target

FEATURE_COLS = [
    "home_form", "home_gf_avg", "home_ga_avg", "home_xg_avg",
    "away_form", "away_gf_avg", "away_ga_avg", "away_xg_avg",
    "h2h_home_winrate",
]
if "home_sentiment" in data.columns:
    FEATURE_COLS += ["home_sentiment", "away_sentiment"]

X = data[FEATURE_COLS].values.astype(np.float32)
y = data["target"].values.astype(np.int64)

# Train/val/test split (70/15/15)

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
X_val,   X_test, y_val,   y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

# Normalise features

scaler  = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val   = scaler.transform(X_val)
X_test  = scaler.transform(X_test)

# Convert to PyTorch tensors

def to_loader(X, y, batch_size=64, shuffle=True):
    ds = TensorDataset(torch.tensor(X), torch.tensor(y))
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle)

train_loader = to_loader(X_train, y_train, shuffle=True)
val_loader   = to_loader(X_val,   y_val,   shuffle=False)
test_loader  = to_loader(X_test,  y_test,  shuffle=False)

# Define the neural network

class MatchPredictor(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(64, 32),
            nn.ReLU(),

            nn.Linear(32, 3)   # 3 outputs: Home Win, Draw, Away Win
        )

    def forward(self, x):
        return self.net(x)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model     = MatchPredictor(input_dim=len(FEATURE_COLS)).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)

# Training loop

EPOCHS     = 100
best_val   = float("inf")
best_state = None
history    = {"train_loss": [], "val_loss": [], "val_acc": []}

for epoch in range(EPOCHS):
    # Train
    model.train()
    train_loss = 0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        loss = criterion(model(X_batch), y_batch)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    train_loss /= len(train_loader)

    # Validate
    model.eval()
    val_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            out      = model(X_batch)
            val_loss += criterion(out, y_batch).item()
            correct  += (out.argmax(1) == y_batch).sum().item()
            total    += y_batch.size(0)
    val_loss /= len(val_loader)
    val_acc   = correct / total

    scheduler.step(val_loss)
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    if val_loss < best_val:
        best_val   = val_loss
        best_state = model.state_dict().copy()

    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1:3d}/{EPOCHS}  train_loss: {train_loss:.4f}  val_loss: {val_loss:.4f}  val_acc: {val_acc:.3f}")

# Evaluate on test set

model.load_state_dict(best_state)
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        preds = model(X_batch.to(device)).argmax(1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(y_batch.numpy())

print("\n── Test Set Results ──────────────────────────────")
print(classification_report(all_labels, all_preds, target_names=["Home Win", "Draw", "Away Win"]))

# Save model & scaler

import pickle
torch.save(best_state, "../Data/model.pth")
with open("../Data/scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)
with open("../Data/feature_cols.pkl", "wb") as f:
    pickle.dump(FEATURE_COLS, f)
print("Model saved!")

# Prediction function (give it two team names)

def predict_match(home_team, away_team):
    # Get last 5 matches for each team from the dataset
    def get_team_stats(team, is_home):
        home_m = data[data["HomeTeam"] == team].tail(5)
        away_m = data[data["AwayTeam"] == team].tail(5)
        all_m  = pd.concat([home_m, away_m]).sort_values("Date").tail(5)
        prefix = "home" if is_home else "away"
        return {
            f"{prefix}_form"   : all_m[f"{prefix}_form"].mean(),
            f"{prefix}_gf_avg" : all_m[f"{prefix}_gf_avg"].mean(),
            f"{prefix}_ga_avg" : all_m[f"{prefix}_ga_avg"].mean(),
            f"{prefix}_xg_avg" : all_m[f"{prefix}_xg_avg"].mean(),
        }

    home_stats = get_team_stats(home_team, is_home=True)
    away_stats = get_team_stats(away_team, is_home=False)

    features = {**home_stats, **away_stats, "h2h_home_winrate": 0.4}
    if "home_sentiment" in FEATURE_COLS:
        features["home_sentiment"] = 0.0
        features["away_sentiment"] = 0.0

    X_input = np.array([[features[c] for c in FEATURE_COLS]], dtype=np.float32)
    X_input = scaler.transform(X_input)

    model.eval()
    with torch.no_grad():
        probs = torch.softmax(model(torch.tensor(X_input).to(device)), dim=1).cpu().numpy()[0]

    labels = ["Home Win", "Draw", "Away Win"]
    print(f"\n⚽ {home_team} vs {away_team}")
    print(f"{'─'*35}")
    for label, prob in zip(labels, probs):
        bar = "█" * int(prob * 30)
        print(f"  {label:<12} {prob*100:5.1f}%  {bar}")
    print(f"\n  Prediction: {labels[probs.argmax()]}")

# Test it
predict_match("Arsenal", "Man City")